# Train an ASL Letter + Number Recognizer (A–Z, 0–9)

Trains a **small classifier** on **MediaPipe hand landmarks** for the sign-language learning app.

- **No model is trained from scratch** and **MediaPipe is reused as-is** — we only train a tiny `scikit-learn` classifier over 21 hand-landmark points.
- **No self-recorded data required** — it trains on two public Kaggle datasets:
  - Letters A–Z: `srisahithis/american-sign-language-a-z-dataset-hand-landmarks`
  - Numbers 0–9: `rayeed045/american-sign-language-digit-dataset`
- **Output:** `asl_landmark_model.joblib` + `labels.json` you download and load in the backend.
- **Runtime:** CPU is fine (no GPU needed). Landmark extraction over ~13k images takes ~10–20 min.

> ⚠️ **J** and **Z** are motion letters; a static classifier only sees one frame — handle them specially in the app (animated reference). Some letters/numbers share handshapes (2≈V, 6≈W, 9≈F, 0≈O) — the app disambiguates by knowing whether it expects a letter or a digit.


## 1. Install dependencies

In [ ]:
!pip -q install mediapipe opencv-python-headless scikit-learn pandas joblib matplotlib tqdm kaggle

## 2. Get the data from Kaggle

You need a Kaggle API token: go to **kaggle.com → Settings → Create New Token** to download `kaggle.json`, then run the cell and upload it.

*(Alternatively, skip this cell and upload/unzip the datasets yourself into `data/letters` and `data/digits` using the Files panel.)*


In [ ]:
from google.colab import files
import os

print("Upload your kaggle.json (from kaggle.com -> Settings -> Create New Token):")
files.upload()  # choose kaggle.json

os.makedirs('/root/.kaggle', exist_ok=True)
os.replace('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d srisahithis/american-sign-language-a-z-dataset-hand-landmarks -p data/letters --unzip
!kaggle datasets download -d rayeed045/american-sign-language-digit-dataset -p data/digits --unzip
print("Downloaded. Peek at the folder layout below.")

In [ ]:
# Inspect the extracted structure so you can confirm the label folders.
for root, dirs, fnames in os.walk('data'):
    depth = root.count(os.sep)
    if depth <= 2:
        print(root, '->', dirs[:15], f'({len(fnames)} files)')

## 3. Configure paths + label mapping

Each image's **immediate parent folder name** is used as its label (works for the usual `.../A/img.jpg`, `.../5/img.jpg`, `asl_alphabet_train/A/...` layouts). Edit `DATA_ROOTS` if your folders differ.


In [ ]:
import glob, cv2, numpy as np, pandas as pd
from tqdm.auto import tqdm

DATA_ROOTS = ["data/letters", "data/digits"]   # edit if your paths differ
IMG_EXTS   = (".jpg", ".jpeg", ".png", ".bmp")
CACHE_CSV  = "landmarks.csv"

WORD_TO_DIGIT = {"zero":"0","one":"1","two":"2","three":"3","four":"4",
                 "five":"5","six":"6","seven":"7","eight":"8","nine":"9"}
SKIP = {"space","del","delete","nothing","blank","background"}

def normalize_label(name):
    n = name.strip().lower()
    if n in SKIP: return None
    if n in WORD_TO_DIGIT: return WORD_TO_DIGIT[n]
    n = n.replace("digit_","").replace("letter_","").replace("sign_","")
    if len(n) == 1 and (n.isalpha() or n.isdigit()):
        return n.upper()
    return None   # unknown folder name -> skip

def list_images(root):
    out = []
    for ext in IMG_EXTS:
        out += glob.glob(os.path.join(root, "**", "*" + ext), recursive=True)
    return out

## 4. Extract + normalize MediaPipe landmarks

In [ ]:
import mediapipe as mp
mp_hands = mp.solutions.hands

def normalize_landmarks(pts):
    # pts: (21,3). Translate wrist->origin, scale to unit -> translation/scale invariant. Returns 63-vector.
    pts = pts.astype(np.float32).copy()
    pts -= pts[0]                       # landmark 0 = wrist
    scale = np.linalg.norm(pts, axis=1).max()
    if scale > 1e-6:
        pts /= scale
    return pts.flatten()

def extract_from_image(img_bgr, hands):
    res = hands.process(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    if not res.multi_hand_landmarks:
        return None
    hand = res.multi_hand_landmarks[0]
    handed = res.multi_handedness[0].classification[0].label if res.multi_handedness else "Right"
    pts = np.array([[lm.x, lm.y, lm.z] for lm in hand.landmark], dtype=np.float32)
    if handed == "Left":               # mirror so every sample looks right-handed
        pts[:, 0] = -pts[:, 0]
    return normalize_landmarks(pts)

rows = []
with mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5) as hands:
    for root in DATA_ROOTS:
        imgs = list_images(root)
        print(f"{root}: {len(imgs)} images")
        for path in tqdm(imgs, desc=root):
            label = normalize_label(os.path.basename(os.path.dirname(path)))
            if label is None:
                continue
            img = cv2.imread(path)
            if img is None:
                continue
            feat = extract_from_image(img, hands)
            if feat is not None:
                rows.append([label, *feat.tolist()])

cols = ["label"] + [f"{a}{i}" for i in range(21) for a in ("x", "y", "z")]
df = pd.DataFrame(rows, columns=cols)
df.to_csv(CACHE_CSV, index=False)
print("Extracted", len(df), "samples across", df["label"].nunique(), "classes")
print(df["label"].value_counts().sort_index())

## 5. Train the small classifier

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

df = pd.read_csv(CACHE_CSV)
X = df.drop(columns=["label"]).values
y = df["label"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

candidates = {
    "SVM": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10, gamma="scale", probability=True)),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
}
scores = {}
for name, clf in candidates.items():
    clf.fit(X_tr, y_tr)
    scores[name] = clf.score(X_te, y_te)
    print(f"{name} test accuracy: {scores[name]:.3f}")

best = max(scores, key=scores.get)
model = candidates[best]
print(f"\nChosen model: {best}")
print(classification_report(y_te, model.predict(X_te)))

labels_sorted = sorted(np.unique(y))
cm = confusion_matrix(y_te, model.predict(X_te), labels=labels_sorted)
fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay(cm, display_labels=labels_sorted).plot(ax=ax, xticks_rotation="vertical", colorbar=False)
plt.title(f"Confusion matrix ({best})"); plt.show()

## 6. Save + download the model

In [ ]:
import joblib, json
joblib.dump(model, "asl_landmark_model.joblib")
json.dump(sorted(np.unique(y).tolist()), open("labels.json", "w"))
print("Saved asl_landmark_model.joblib and labels.json")

try:
    from google.colab import files
    files.download("asl_landmark_model.joblib")
    files.download("labels.json")
except Exception as e:
    print("Download manually from the Files panel.", e)

## 7. Quick inference test

The same `extract_from_image` + `normalize_landmarks` used here is what the backend runs at inference time: MediaPipe landmarks → normalize → `model.predict`.


In [ ]:
def predict_sign(image_path, model, top_k=3):
    img = cv2.imread(image_path)
    with mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5) as hands:
        feat = extract_from_image(img, hands)
    if feat is None:
        return "no hand detected", []
    proba = model.predict_proba([feat])[0]
    classes = model.classes_
    order = np.argsort(proba)[::-1][:top_k]
    return classes[order[0]], [(classes[i], float(proba[i])) for i in order]

# Example — replace with a real path from your data:
# label, top = predict_sign("data/digits/5/xxxx.jpg", model)
# print("Predicted:", label, "| top:", top)

## 8. Using this model in the app

At runtime the backend does exactly what Section 7 does per frame:

```python
import joblib, json, numpy as np
model  = joblib.load("asl_landmark_model.joblib")
labels = json.load(open("labels.json"))
# feat = normalize_landmarks(<21x3 MediaPipe landmarks for the current frame>)
# pred = model.predict([feat])[0]           # a letter 'A'..'Z' or digit '0'..'9'
```

Then:
- **Stability gate:** only accept `pred` after it repeats across N consecutive frames above a probability threshold.
- **Word / multi-digit number:** advance a pointer over the target sequence (e.g. `CAT` → C,A,T; `25` → 2,5), matching one target at a time.
- **Letter vs digit mode:** when the app expects a digit, restrict/prefer digit classes to avoid handshape overlaps (2≈V, 6≈W, 9≈F, 0≈O).
- **J / Z:** motion letters — don't grade from a single frame; use the animated-reference/relaxed handling described in the PRD.

**Optional accuracy tuning (only if needed):** capture a few of your own samples per sign with a webcam → MediaPipe → append rows to `landmarks.csv` (same 63 columns) → re-run Section 5. Not required to start.
